# 05 DCF, SOTP, Sensitivities and Memo

Final question: after the post-earnings pullback, does Oracle's current
price pay too little, too much, or about enough for the OCI backlog?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    DcfInputs,
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    discounted_cash_flow,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/oracle_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/oracle_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
options = pd.read_csv(DATA_DIR / "option_scenarios.csv")
premiums = pd.read_csv(DATA_DIR / "market_premiums.csv").set_index("case")
cases = ["bear", "base", "bull"]

def case_table(segment):
    return (
        assumptions[assumptions["segment"].eq(segment)]
        .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
        .reindex(cases)
    )

def segment_value(table):
    return (
        table["FY2030 revenue"]
        * table["EBITDA margin"]
        * table["EV/EBITDA multiple"]
        * table["discount factor"]
    )

In [ ]:
segment_tables = {
    segment: case_table(segment)
    for segment in ["OCI", "Cloud Apps", "Database Support", "Services Hardware"]
}
for table in segment_tables.values():
    table["ebitda_usd_bn"] = table["FY2030 revenue"] * table["EBITDA margin"]
    table["selected_value_usd_bn"] = segment_value(table)

option_values = {}
for option_name, group in options.groupby("option"):
    scenarios = [
        OptionScenario(row.case, row.probability, row.value_usd_bn)
        for row in group.itertuples(index=False)
    ]
    option_values[option_name] = probability_weighted_option_value(scenarios)
option_values

In [ ]:
capital = (
    assumptions[assumptions["segment"].eq("Capital")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
)
sotp_rows = []
for case in cases:
    segments = [
        SegmentValuation("OCI", segment_tables["OCI"].loc[case, "selected_value_usd_bn"], "FY2030 EV/EBITDA PV"),
        SegmentValuation("Cloud Apps", segment_tables["Cloud Apps"].loc[case, "selected_value_usd_bn"], "FY2030 EV/EBITDA PV"),
        SegmentValuation("Database Support", segment_tables["Database Support"].loc[case, "selected_value_usd_bn"], "FY2030 EV/EBITDA PV"),
        SegmentValuation("Services Hardware", segment_tables["Services Hardware"].loc[case, "selected_value_usd_bn"], "FY2030 EV/EBITDA PV"),
        SegmentValuation("AI capacity conversion", option_values["AI capacity conversion"], "Probability weighted option"),
        SegmentValuation("Capex/customer stress", option_values["Capex and customer concentration stress"], "Probability weighted option"),
    ]
    bridge = sotp_equity_value(
        SotpInputs(
            segments=segments,
            cash=capital.loc["base", "cash"],
            debt=capital.loc["base", "debt and lease liabilities"],
            dilution=capital.loc["base", "dilution"],
            fully_diluted_shares=capital.loc["base", "fully diluted shares"],
        )
    )
    premium = premiums.loc[case]
    market = market_premium_value(
        bridge.equity_value,
        MarketPremiumInputs(
            ai_scarcity_premium=premium.ai_scarcity_premium,
            ipo_scarcity_premium=premium.mega_cap_liquidity_premium,
            strategic_asset_premium=premium.strategic_asset_premium,
            governance_discount=premium.governance_discount,
            execution_haircut=premium.execution_haircut,
        ),
    )
    sotp_rows.append(
        {
            "case": case,
            "segment_value_usd_bn": bridge.segment_value,
            "equity_value_before_premium_usd_bn": bridge.equity_value,
            "net_premium_discount": market.net_premium,
            "market_adjusted_equity_value_usd_bn": market.market_value,
            "value_per_share_usd": market.market_value / capital.loc["base", "fully diluted shares"],
        }
    )
sotp = pd.DataFrame(sotp_rows).set_index("case")
sotp

In [ ]:
market = (
    assumptions[assumptions["segment"].eq("Market")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
)
current_price = market.loc["base", "current price"]
current_equity_value = current_price * capital.loc["base", "fully diluted shares"]
current_enterprise_value = (
    current_equity_value
    - capital.loc["base", "cash"]
    + capital.loc["base", "debt and lease liabilities"]
    + capital.loc["base", "dilution"]
)
current_context = pd.DataFrame(
    [
        {
            "current_price": current_price,
            "current_equity_value_usd_bn": current_equity_value,
            "current_enterprise_value_usd_bn": current_enterprise_value,
            "base_value_per_share_usd": sotp.loc["base", "value_per_share_usd"],
            "base_upside_downside": sotp.loc["base", "value_per_share_usd"] / current_price - 1,
        }
    ]
)
current_context

In [ ]:
base = "base"
base_premium = premiums.loc[base]
base_net_premium = (
    base_premium.ai_scarcity_premium
    + base_premium.mega_cap_liquidity_premium
    + base_premium.strategic_asset_premium
    + base_premium.governance_discount
    + base_premium.execution_haircut
)
required_equity_before_premium = current_equity_value / (1 + base_net_premium)
required_segment_value = (
    required_equity_before_premium
    - (capital.loc["base", "cash"] - capital.loc["base", "debt and lease liabilities"])
    + capital.loc["base", "dilution"]
)
non_oci_value = sum(
    segment_tables[segment].loc[base, "selected_value_usd_bn"]
    for segment in ["Cloud Apps", "Database Support", "Services Hardware"]
) + sum(option_values.values())
required_oci_present_value = required_segment_value - non_oci_value
oci_base = segment_tables["OCI"].loc[base]
required_oci_revenue = (
    required_oci_present_value
    / oci_base["discount factor"]
    / oci_base["EBITDA margin"]
    / oci_base["EV/EBITDA multiple"]
)
priced_in = pd.DataFrame(
    [
        {
            "current_price": current_price,
            "required_OCI_PV_usd_bn": required_oci_present_value,
            "required_FY2030_OCI_revenue_usd_bn": required_oci_revenue,
            "base_case_FY2030_OCI_revenue_usd_bn": oci_base["FY2030 revenue"],
            "discount_to_base_OCI_revenue": required_oci_revenue / oci_base["FY2030 revenue"] - 1,
        }
    ]
)
priced_in

In [ ]:
dcf_inputs = {
    "bear": DcfInputs(free_cash_flow=12.0, growth_rate=0.08, discount_rate=0.105, terminal_growth_rate=0.025, years=7),
    "base": DcfInputs(free_cash_flow=24.0, growth_rate=0.13, discount_rate=0.095, terminal_growth_rate=0.030, years=7),
    "bull": DcfInputs(free_cash_flow=38.0, growth_rate=0.16, discount_rate=0.090, terminal_growth_rate=0.035, years=7),
}
dcf_rows = []
for case, inputs in dcf_inputs.items():
    result = discounted_cash_flow(inputs)
    equity_value = (
        result.enterprise_value
        + capital.loc["base", "cash"]
        - capital.loc["base", "debt and lease liabilities"]
        - capital.loc["base", "dilution"]
    )
    dcf_rows.append(
        {
            "case": case,
            "dcf_enterprise_value_usd_bn": result.enterprise_value,
            "dcf_equity_value_usd_bn": equity_value,
            "dcf_value_per_share_usd": equity_value / capital.loc["base", "fully diluted shares"],
        }
    )
dcf_cross_check = pd.DataFrame(dcf_rows).set_index("case")
dcf_cross_check

In [ ]:
reverse_grid = build_sensitivity_grid(
    row_values=[95, 120, 145, 166, 190, 220],
    column_values=[0.25, 0.30, 0.34, 0.38, 0.42],
    formula=lambda revenue, margin: revenue * margin * 14.0 * 0.68,
    row_name="FY2030 OCI revenue (USD bn)",
    column_name="OCI EBITDA margin",
)
reverse_grid

In [ ]:
football = pd.DataFrame(
    {
        "method": [
            "OCI FY2030 EV/EBITDA PV",
            "Cloud apps",
            "Database support",
            "Options and stress",
            "SOTP before premium",
            "Market-adjusted SOTP",
            "DCF cross-check",
        ],
        "low": [
            segment_tables["OCI"].loc["bear", "selected_value_usd_bn"],
            segment_tables["Cloud Apps"].loc["bear", "selected_value_usd_bn"],
            segment_tables["Database Support"].loc["bear", "selected_value_usd_bn"],
            min(option_values.values()),
            sotp.loc["bear", "equity_value_before_premium_usd_bn"],
            sotp.loc["bear", "market_adjusted_equity_value_usd_bn"],
            dcf_cross_check.loc["bear", "dcf_equity_value_usd_bn"],
        ],
        "high": [
            segment_tables["OCI"].loc["bull", "selected_value_usd_bn"],
            segment_tables["Cloud Apps"].loc["bull", "selected_value_usd_bn"],
            segment_tables["Database Support"].loc["bull", "selected_value_usd_bn"],
            max(option_values.values()),
            sotp.loc["bull", "equity_value_before_premium_usd_bn"],
            sotp.loc["bull", "market_adjusted_equity_value_usd_bn"],
            dcf_cross_check.loc["bull", "dcf_equity_value_usd_bn"],
        ],
    }
)
ax = football.plot.barh(x="method", y=["low", "high"], figsize=(9, 5), title="Oracle Valuation Football Field")
ax.set_xlabel("USD bn")
plt.tight_layout()
football

## Research Memo Draft

At roughly $184 per share, Oracle is no longer just a mature database
software underwriting. The current price appears to imply about $124B of
FY2030 OCI revenue under the base margin, multiple and discount framework,
below the seeded base case of $166B, but only after accepting meaningful
funding, dilution and execution haircuts.

What has to go right:

- RPO must convert into revenue rather than remain a financing-heavy backlog headline.
- OCI margins need to move toward management's mature 30-40 percent frame as capacity fills.
- Customer prepayments and bring-your-own-hardware terms must reduce cash-flow strain.
- Database support and applications must remain durable enough to fund the transition.
- Debt, leases and equity issuance must not absorb the equity upside from OCI growth.

What breaks first in downside is not demand; it is conversion economics. If
capacity slips, customer concentration rises, or capex keeps outrunning
revenue recognition, the equity can de-rate even while reported RPO remains
large. The current setup is best treated as screen-grade and worth
re-underwriting when the FY2026 10-K, customer concentration detail and
FY2027 capex funding mix are normalized.